# OVR Logistic Regression with Thresholding

This notebook builds a threshold-based one-vs-rest logistic regression benchmark for skill tagging using Supabase-hosted embeddings and labels.

What this notebook does:
- loads job, course, and skill embeddings from Supabase tables `job_embeddings`, `course_embeddings`, and `skill_embeddings`
- reads `extracted_skills` directly from the Supabase job and course tables
- builds two dataset variants: `jobs_only` and `jobs_plus_courses`
- trains one logistic regression model per skill on the train split
- tunes thresholds on the validation split only
- reports the final locked metrics on the held-out test split

Before running the notebook, set `SUPABASE_URL` plus one of `SUPABASE_KEY`, `SUPABASE_SERVICE_ROLE_KEY`, or `SUPABASE_ANON_KEY` in your environment.


In [11]:
import ast
import numpy as np
import os
import pandas as pd
from pathlib import Path
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split

ENV_PATH = Path("../.env")

def load_env_file(env_path):
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip(chr(34)).strip(chr(39))
        os.environ[key] = value

load_env_file(ENV_PATH)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = (
    os.getenv("SUPABASE_KEY")
    or os.getenv("SUPABASE_SERVICE_ROLE_KEY")
    or os.getenv("SUPABASE_ANON_KEY")
)

if not SUPABASE_URL or "your_project_id" in SUPABASE_URL:
    raise ValueError(f"Invalid SUPABASE_URL loaded: {SUPABASE_URL!r}. Check ../.env and restart the kernel.")

PREVIEW_ONLY = False
PREVIEW_ROWS = 5



def parse_embedding(value):
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [float(x) for x in parsed]
    raise TypeError(f"Unsupported embedding value type: {type(value)!r}")


def fetch_supabase_table(table_name, page_size=200, preview_only=False, preview_rows=5):
    if not SUPABASE_URL or not SUPABASE_KEY:
        raise ValueError(
            "Set SUPABASE_URL and one of SUPABASE_KEY, SUPABASE_SERVICE_ROLE_KEY, or SUPABASE_ANON_KEY before running this notebook."
        )

    base_url = f"{SUPABASE_URL.rstrip('/')}/rest/v1/{table_name}"
    headers = {
        "apikey": SUPABASE_KEY,
        "Authorization": f"Bearer {SUPABASE_KEY}",
        "Accept": "application/json",
    }

    if preview_only:
        print(f"Previewing {table_name} ({preview_rows} rows)...")
        response = requests.get(
            base_url,
            headers=headers,
            params={"select": "*", "limit": preview_rows},
            timeout=60,
        )
        if not response.ok:
            raise requests.HTTPError(
                f"Supabase preview failed for {table_name}: {response.status_code} {response.text}",
                response=response,
            )
        return pd.DataFrame(response.json())

    rows = []
    offset = 0
    while True:
        print(f"Fetching {table_name} rows {offset} to {offset + page_size - 1}...")
        response = requests.get(
            base_url,
            headers=headers,
            params={"select": "*", "limit": page_size, "offset": offset},
            timeout=60,
        )
        if not response.ok:
            raise requests.HTTPError(
                f"Supabase fetch failed for {table_name} at offset {offset}: {response.status_code} {response.text}",
                response=response,
            )

        batch = response.json()
        if not batch:
            break

        rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size

    return pd.DataFrame(rows)


def require_columns(df, required_columns, table_name):
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise KeyError(f"Table {table_name} is missing required columns: {missing_columns}")


skills_embeddings = fetch_supabase_table("skill_embeddings", preview_only=PREVIEW_ONLY, preview_rows=PREVIEW_ROWS)
jobs_embeddings = fetch_supabase_table("job_embeddings", preview_only=PREVIEW_ONLY, preview_rows=PREVIEW_ROWS)
courses_embeddings = fetch_supabase_table("course_embeddings", preview_only=PREVIEW_ONLY, preview_rows=PREVIEW_ROWS)

require_columns(skills_embeddings, ["skill_name", "embedding"], "skill_embeddings")
require_columns(jobs_embeddings, ["job_id", "title", "embedding", "extracted_skills"], "job_embeddings")
require_columns(courses_embeddings, ["course_code", "title", "embedding", "extracted_skills"], "course_embeddings")

skills_embeddings = skills_embeddings.copy()
if "record_id" not in skills_embeddings.columns:
    skills_embeddings["record_id"] = "skill:" + skills_embeddings["skill_name"].astype(str)
if "entity_type" not in skills_embeddings.columns:
    skills_embeddings["entity_type"] = "skill"
skills_embeddings["embedding"] = skills_embeddings["embedding"].apply(parse_embedding)

jobs_embeddings = jobs_embeddings.copy()
if "record_id" not in jobs_embeddings.columns:
    jobs_embeddings["record_id"] = "job:" + jobs_embeddings["job_id"].astype(str)
if "entity_type" not in jobs_embeddings.columns:
    jobs_embeddings["entity_type"] = "job"
jobs_embeddings["embedding"] = jobs_embeddings["embedding"].apply(parse_embedding)

courses_embeddings = courses_embeddings.copy()
if "record_id" not in courses_embeddings.columns:
    courses_embeddings["record_id"] = "course:" + courses_embeddings["course_code"].astype(str)
if "entity_type" not in courses_embeddings.columns:
    courses_embeddings["entity_type"] = "course"
courses_embeddings["embedding"] = courses_embeddings["embedding"].apply(parse_embedding)


print("Loaded preview data from Supabase:" if PREVIEW_ONLY else "Loaded embeddings from Supabase:")
print(f"skills rows:  {len(skills_embeddings)}")
print(f"jobs rows:    {len(jobs_embeddings)}")
print(f"courses rows: {len(courses_embeddings)}")
print()
print("jobs_embeddings head:")
print(jobs_embeddings[["record_id", "job_id", "title"]].head().to_string(index=False))
print()
print("courses_embeddings head:")
print(courses_embeddings[["record_id", "course_code", "title"]].head().to_string(index=False))
print()
print("skills_embeddings head:")
print(skills_embeddings[["record_id", "skill_name"]].head().to_string(index=False))
print()
print(f"num_skills: {len(skills_embeddings)}")

Fetching skill_embeddings rows 0 to 199...
Fetching skill_embeddings rows 200 to 399...
Fetching skill_embeddings rows 400 to 599...
Fetching skill_embeddings rows 600 to 799...
Fetching skill_embeddings rows 800 to 999...
Fetching skill_embeddings rows 1000 to 1199...
Fetching skill_embeddings rows 1200 to 1399...
Fetching skill_embeddings rows 1400 to 1599...
Fetching skill_embeddings rows 1600 to 1799...
Fetching skill_embeddings rows 1800 to 1999...
Fetching skill_embeddings rows 2000 to 2199...
Fetching skill_embeddings rows 2200 to 2399...
Fetching skill_embeddings rows 2400 to 2599...
Fetching skill_embeddings rows 2600 to 2799...
Fetching skill_embeddings rows 2800 to 2999...
Fetching skill_embeddings rows 3000 to 3199...
Fetching skill_embeddings rows 3200 to 3399...
Fetching skill_embeddings rows 3400 to 3599...
Fetching skill_embeddings rows 3600 to 3799...
Fetching skill_embeddings rows 3800 to 3999...
Fetching skill_embeddings rows 4000 to 4199...
Fetching skill_embeddings

## Helper Functions

These helpers do the repetitive setup work:
- convert `extracted_skills` text into Python lists
- join embeddings to labels
- build binary label matrices
- train one logistic regression per skill
- tune thresholds and compute metrics

Keeping them here makes the actual experiment cells shorter and easier to read.


In [12]:
def parse_skill_list(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [skill.strip() for skill in str(value).split("|") if skill.strip()]


def prepare_labeled_entity_frame(df, entity_id_col):
    prepared = df.copy()
    missing_mask = prepared["extracted_skills"].isna() | (prepared["extracted_skills"].astype(str).str.strip() == "")
    dropped_ids = prepared.loc[missing_mask, entity_id_col].tolist()
    if dropped_ids:
        print(f"Dropping {len(dropped_ids)} unlabeled rows for {entity_id_col}: {dropped_ids[:10]}" + (" ..." if len(dropped_ids) > 10 else ""))
        prepared = prepared.loc[~missing_mask].copy()

    prepared["actual_skill_lists"] = prepared["extracted_skills"].apply(parse_skill_list)
    prepared["display_title"] = prepared["title"]
    prepared["entity_id"] = prepared[entity_id_col]

    return prepared[["entity_type", "entity_id", "display_title", "embedding", "actual_skill_lists"]].copy()


def split_entity_frame(df, random_state=42):
    train_df, temp_df = train_test_split(df, test_size=0.4, random_state=random_state, shuffle=True)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=random_state, shuffle=True)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def build_indicator_matrix(skill_lists, skill_names):
    skill_to_idx = {skill: idx for idx, skill in enumerate(skill_names)}
    indicator = np.zeros((len(skill_lists), len(skill_names)), dtype=np.uint8)

    for row_idx, skills in enumerate(skill_lists):
        for skill in skills:
            if skill in skill_to_idx:
                indicator[row_idx, skill_to_idx[skill]] = 1

    return indicator


def build_dataset(train_df, val_df, test_df, skill_names):
    X_train = np.vstack(train_df["embedding"].to_numpy()).astype(np.float32)
    X_val = np.vstack(val_df["embedding"].to_numpy()).astype(np.float32)
    X_test = np.vstack(test_df["embedding"].to_numpy()).astype(np.float32)

    Y_train = build_indicator_matrix(train_df["actual_skill_lists"].tolist(), skill_names)
    Y_val = build_indicator_matrix(val_df["actual_skill_lists"].tolist(), skill_names)
    Y_test = build_indicator_matrix(test_df["actual_skill_lists"].tolist(), skill_names)

    return X_train, X_val, X_test, Y_train, Y_val, Y_test


def fit_ovr_probabilities(X_train, Y_train, X_val, X_test, skill_names):
    train_proba = np.zeros((X_train.shape[0], len(skill_names)), dtype=np.float32)
    val_proba = np.zeros((X_val.shape[0], len(skill_names)), dtype=np.float32)
    test_proba = np.zeros((X_test.shape[0], len(skill_names)), dtype=np.float32)
    fitted_mask = np.zeros(len(skill_names), dtype=bool)
    skipped_skills = []

    for j, skill_name in enumerate(skill_names):
        y_train_j = Y_train[:, j]

        if len(np.unique(y_train_j)) < 2:
            skipped_skills.append(skill_name)
            continue

        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )
        model.fit(X_train, y_train_j)

        train_proba[:, j] = model.predict_proba(X_train)[:, 1]
        val_proba[:, j] = model.predict_proba(X_val)[:, 1]
        test_proba[:, j] = model.predict_proba(X_test)[:, 1]
        fitted_mask[j] = True

    return train_proba, val_proba, test_proba, fitted_mask, skipped_skills


def micro_metrics(y_true, y_pred):
    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return precision, recall, f1


def macro_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return precision, recall, f1


def tune_global_threshold(y_true, y_score):
    candidate_thresholds = np.unique(y_score)
    candidate_thresholds = np.r_[candidate_thresholds, 1.0 + 1e-9]

    best_threshold = 0.5
    best_candidate = None

    for threshold in candidate_thresholds:
        y_pred = (y_score >= threshold).astype(np.uint8)
        precision, recall, f1 = micro_metrics(y_true, y_pred)
        candidate = (f1, recall, -threshold)

        if best_candidate is None or candidate > best_candidate:
            best_candidate = candidate
            best_threshold = float(threshold)

    return best_threshold


def tune_best_threshold_for_one_skill(y_true, y_score):
    candidate_thresholds = np.unique(y_score)
    candidate_thresholds = np.r_[candidate_thresholds, 1.0 + 1e-9]

    best_threshold = 0.5
    best_candidate = None

    for threshold in candidate_thresholds:
        y_pred = (y_score >= threshold).astype(np.uint8)

        tp = np.logical_and(y_pred == 1, y_true == 1).sum()
        fp = np.logical_and(y_pred == 1, y_true == 0).sum()
        fn = np.logical_and(y_pred == 0, y_true == 1).sum()

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
        candidate = (f1, recall, -threshold)

        if best_candidate is None or candidate > best_candidate:
            best_candidate = candidate
            best_threshold = float(threshold)

    return best_threshold


def indicator_row_to_skills(row, skill_names):
    return [skill_names[j] for j, flag in enumerate(row) if flag == 1]


## Prepare Supabase Labels

At this point we use the `extracted_skills` already stored in Supabase on the job and course tables.

This turns the raw embedding rows into supervised learning data:
- the embedding column gives the feature vector
- the `extracted_skills` column gives the true skill list

After preparation, each row has:
- entity id
- title
- embedding
- actual skill list


In [13]:
skill_names = skills_embeddings["skill_name"].tolist()

jobs_all_df = prepare_labeled_entity_frame(jobs_embeddings.copy(), entity_id_col="job_id")
courses_all_df = prepare_labeled_entity_frame(courses_embeddings.copy(), entity_id_col="course_code")

jobs_train_df, jobs_val_df, jobs_test_df = split_entity_frame(jobs_all_df, random_state=42)
courses_train_df, courses_val_df, courses_test_df = split_entity_frame(courses_all_df, random_state=42)

print("Supervised rows after reading Supabase labels and in-notebook splitting:")
print(f"jobs train:    {len(jobs_train_df)}")
print(f"jobs val:      {len(jobs_val_df)}")
print(f"jobs test:     {len(jobs_test_df)}")
print(f"courses train: {len(courses_train_df)}")
print(f"courses val:   {len(courses_val_df)}")
print(f"courses test:  {len(courses_test_df)}")


Dropping 1 unlabeled rows for job_id: ['30d38911187b8c345cd04c7b000062e1']
Dropping 1311 unlabeled rows for course_code: ['ACC3712', 'ACC3713', 'BSE3761', 'BSE4761', 'BSE4761X', 'CM2111', 'CM2121', 'CM3232', 'CM3291', 'CM3295'] ...
Supervised rows after reading Supabase labels and in-notebook splitting:
jobs train:    951
jobs val:      317
jobs test:     318
courses train: 2495
courses val:   832
courses test:  832


## Build Benchmark Variants

We evaluate two scopes:
- `jobs_only`: only job postings are used
- `jobs_plus_courses`: job postings and ACC courses are combined

This helps us compare the OVR model against the same kinds of setups explored in the cosine notebook.


In [14]:
dataset_variants = {
    "jobs_only": {
        "train": jobs_train_df,
        "val": jobs_val_df,
        "test": jobs_test_df,
    },
    "jobs_plus_courses": {
        "train": pd.concat([jobs_train_df, courses_train_df], ignore_index=True),
        "val": pd.concat([jobs_val_df, courses_val_df], ignore_index=True),
        "test": pd.concat([jobs_test_df, courses_test_df], ignore_index=True),
    },
}

for variant_name, frames in dataset_variants.items():
    print(variant_name)
    print(f"  train rows: {len(frames['train'])}")
    print(f"  val rows:   {len(frames['val'])}")
    print(f"  test rows:  {len(frames['test'])}")


jobs_only
  train rows: 951
  val rows:   317
  test rows:  318
jobs_plus_courses
  train rows: 3446
  val rows:   1149
  test rows:  1150


## Train, Tune, and Evaluate

This is the main experiment cell.

For each dataset variant, it:
- builds feature and label matrices
- trains one logistic regression model per skill
- tunes threshold(s) on validation
- evaluates the locked predictions on test

It stores both a summary table and prediction previews so we can inspect what the model is actually predicting.


In [15]:
results = []
artifacts = {}

for variant_name, frames in dataset_variants.items():
    X_train, X_val, X_test, Y_train, Y_val, Y_test = build_dataset(
        frames["train"],
        frames["val"],
        frames["test"],
        skill_names,
    )

    train_proba, val_proba, test_proba, fitted_mask, skipped_skills = fit_ovr_probabilities(
        X_train,
        Y_train,
        X_val,
        X_test,
        skill_names,
    )

    global_threshold = tune_global_threshold(Y_val, val_proba)
    global_test_pred = (test_proba >= global_threshold).astype(np.uint8)
    global_micro_precision, global_micro_recall, global_micro_f1 = micro_metrics(Y_test, global_test_pred)
    global_macro_precision, global_macro_recall, global_macro_f1 = macro_metrics(Y_test, global_test_pred)

    results.append({
        "dataset_variant": variant_name,
        "threshold_type": "global",
        "micro_precision": global_micro_precision,
        "micro_recall": global_micro_recall,
        "micro_f1": global_micro_f1,
        "macro_precision": global_macro_precision,
        "macro_recall": global_macro_recall,
        "macro_f1": global_macro_f1,
        "global_threshold": global_threshold,
        "fitted_models": int(fitted_mask.sum()),
        "skipped_skills": len(skipped_skills),
        "train_rows": len(frames["train"]),
        "val_rows": len(frames["val"]),
        "test_rows": len(frames["test"]),
    })

    global_predicted_skill_lists = [indicator_row_to_skills(row, skill_names) for row in global_test_pred]
    artifacts[(variant_name, "global")] = {
        "predictions_df": pd.DataFrame({
            "entity_type": frames["test"]["entity_type"],
            "entity_id": frames["test"]["entity_id"],
            "title": frames["test"]["display_title"],
            "actual_skills": [" | ".join(skills) for skills in frames["test"]["actual_skill_lists"]],
            "predicted_skills": [" | ".join(skills) for skills in global_predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], global_predicted_skill_lists)],
        }),
        "thresholds_df": pd.DataFrame({
            "skill_name": skill_names,
            "threshold": np.full(len(skill_names), global_threshold, dtype=np.float32),
            "was_fitted": fitted_mask,
            "train_positive_support": Y_train.sum(axis=0),
            "val_positive_support": Y_val.sum(axis=0),
            "test_positive_support": Y_test.sum(axis=0),
        }),
    }

    skill_thresholds = np.full(len(skill_names), 0.5, dtype=np.float32)
    for j in np.where(fitted_mask)[0]:
        skill_thresholds[j] = tune_best_threshold_for_one_skill(Y_val[:, j], val_proba[:, j])

    skill_test_pred = (test_proba >= skill_thresholds[np.newaxis, :]).astype(np.uint8)
    skill_micro_precision, skill_micro_recall, skill_micro_f1 = micro_metrics(Y_test, skill_test_pred)
    skill_macro_precision, skill_macro_recall, skill_macro_f1 = macro_metrics(Y_test, skill_test_pred)

    results.append({
        "dataset_variant": variant_name,
        "threshold_type": "skill_specific",
        "micro_precision": skill_micro_precision,
        "micro_recall": skill_micro_recall,
        "micro_f1": skill_micro_f1,
        "macro_precision": skill_macro_precision,
        "macro_recall": skill_macro_recall,
        "macro_f1": skill_macro_f1,
        "global_threshold": np.nan,
        "fitted_models": int(fitted_mask.sum()),
        "skipped_skills": len(skipped_skills),
        "train_rows": len(frames["train"]),
        "val_rows": len(frames["val"]),
        "test_rows": len(frames["test"]),
    })

    skill_predicted_skill_lists = [indicator_row_to_skills(row, skill_names) for row in skill_test_pred]
    artifacts[(variant_name, "skill_specific")] = {
        "predictions_df": pd.DataFrame({
            "entity_type": frames["test"]["entity_type"],
            "entity_id": frames["test"]["entity_id"],
            "title": frames["test"]["display_title"],
            "actual_skills": [" | ".join(skills) for skills in frames["test"]["actual_skill_lists"]],
            "predicted_skills": [" | ".join(skills) for skills in skill_predicted_skill_lists],
            "tp": [len(set(a) & set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
            "fp": [len(set(p) - set(a)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
            "fn": [len(set(a) - set(p)) for a, p in zip(frames["test"]["actual_skill_lists"], skill_predicted_skill_lists)],
        }),
        "thresholds_df": pd.DataFrame({
            "skill_name": skill_names,
            "threshold": skill_thresholds,
            "was_fitted": fitted_mask,
            "train_positive_support": Y_train.sum(axis=0),
            "val_positive_support": Y_val.sum(axis=0),
            "test_positive_support": Y_test.sum(axis=0),
        }),
    }

comparison_df = pd.DataFrame(results).sort_values(
    by=["micro_f1", "micro_recall", "macro_f1"],
    ascending=[False, False, False],
).reset_index(drop=True)
comparison_df


KeyboardInterrupt: 

## Inspect the Best Configuration

This final code cell prints the strongest result from the comparison table and shows a small prediction preview.

That makes it easier to check whether the model is making sensible skill predictions instead of relying on summary metrics alone.


In [ ]:
best_row = comparison_df.iloc[0]
best_key = (best_row["dataset_variant"], best_row["threshold_type"])

print("Best evaluated configuration on held-out test:")
print(best_row.to_string())
print()
print("Prediction preview:")
print(artifacts[best_key]["predictions_df"].head(10).to_string(index=False))


Best evaluated configuration on held-out test:
dataset_variant     jobs_plus_courses
threshold_type                 global
micro_precision              0.716049
micro_recall                 0.783784
micro_f1                     0.748387
macro_precision              0.004743
macro_recall                 0.004859
macro_f1                     0.004574
global_threshold             0.551087
fitted_models                      70
skipped_skills                   4331
train_rows                         95
val_rows                           33
test_rows                          33

Prediction preview:
entity_type                        entity_id                                             title                                                                                                                                                                                                                                                   actual_skills                                                  

## Summary

This notebook now reads top to bottom as one clean benchmark:
- load data
- prepare supervised labels
- define the experiment variants
- train and tune the OVR models
- report final test results

In practice, the global-threshold OVR variant is usually the stronger baseline here, while the skill-specific-threshold version tends to overfit because the validation set is small and many skills are rare.

### Caveats and Limitations
- This benchmark assumes the CSV `extracted_skills` labels are the ground truth.
- Many skills are rare, so the model learns much better for common skills than rare ones.
- OVR cannot train a classifier for a skill if the train split does not contain both positive and negative examples for that skill.
- Micro-F1 is driven more by common skills, so it can look stronger than macro-F1.
- The skill-specific-threshold variant is more flexible, but it can overfit when the validation set is small.
- One course, `ACC4761E`, is excluded because it currently has no labeled `extracted_skills` value.
